![Image](./snapshot_strategy.png)

1차에서 수행한 100개의 질문을 제외하고 다시 랜덤하게 100개 추출


In [ ]:
# pip install openpyxl 

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import psycopg2
import pandas as pd
import numpy as np
import pickle
import lib.preprocess.preprocess as pp
import lib.preprocess.SectionExtractor as se
import re
import datetime
import pandas as pd
import re
import numpy as np
from sklearn import metrics
import os


In [3]:
htmlp = pp.HTMLParser()
codep = pp.CodeSectionParser()
ts = se.SectionExtractor()

In [4]:
file_list = os.listdir('.')

In [5]:
file_list = [x for x in file_list if x.endswith('snapshop2_sample.csv')]

In [6]:
df_q = pd.DataFrame()
for file in file_list :
    df_q = pd.concat([df_q, pd.read_csv(file, index_col=False)], axis = 0)

In [7]:
df_q[['id', 'question']]

,id,question
0,70264389,<Title>Population pyramid with seaborn python<...
1,70274885,"<Title>insert or update on table ""django_admin..."
2,70313318,<Title>perform upsert operation on postgres li...
3,70540832,<Title>Python/SQL - Connecting to different da...
4,70693775,<Title>How to pass an object to a process crea...
...,...,...
103,70731352,<Title>Python str not list</Title>. <Question>...
104,74047007,<Title>How to detect black contour in image us...
105,78840333,<Title>How to get every combination possible i...
106,71281263,<Title>Loop through a directory and add filena...


In [8]:
path_list = [f'../golden_dataset/{x}' for x in ['2nd', '3rd', '5th']]

In [9]:
a_list = []
for path in path_list : 
    a_list.append([f'{path}/{x}' for x in os.listdir(path) if x.endswith('.xlsx') and not x.startswith('~')])


In [10]:
mapping = {'Basic': '<Difficulty Level>0</Difficulty Level>', 
           'Intermediate': '<Difficulty Level>1</Difficulty Level>', 
           'Advanced' : '<Difficulty Level>2</Difficulty Level>'}

In [23]:
tot_df = pd.DataFrame()
for a in a_list:
    df_0 = pd.read_excel(f'{a[0]}', engine='openpyxl')
    df_1 = pd.read_excel(f'{a[1]}', engine='openpyxl')
    df_2 = pd.read_excel(f'{a[2]}', engine='openpyxl')
    df_3 = pd.read_excel(f'{a[3]}', engine='openpyxl')

    df_0 = df_0[['id', 'answer']].rename(columns={'answer' : 'a_jh'})
    df_1 = df_1[['id', 'answer']].rename(columns={'answer' : 'a_hj'})
    df_2 = df_2[['id', 'answer']].rename(columns={'answer' : 'a_jw'})
    df_3 = df_3[['id', 'answer']].rename(columns={'answer' : 'a_mk'})

    df_0['a_jh'] = df_0['a_jh'].map(mapping)
    df_1['a_hj'] = df_1['a_hj'].map(mapping)
    df_2['a_jw'] = df_2['a_jw'].map(mapping)
    df_3['a_mk'] = df_3['a_mk'].map(mapping)

    df_m = df_0.merge(df_1, on='id') \
                .merge(df_2, on='id') \
                .merge(df_3, on='id')
    
    # df_m['sum'] = (df_m['a_jh']==df_m['a_hj'])&(df_m['a_hj'] ==df_m['a_jw']) & (df_m['a_jw']==df_m['a_mk'])
    # df_c = df_m[df_m['sum'] ==True]
    tot_df = pd.concat([tot_df, df_m], axis = 0)

In [27]:
tot_df = tot_df.melt(id_vars = 'id' , value_vars= ['a_jh', 'a_hj', 'a_jw', 'a_mk'], var_name='annotator', value_name='answer')

In [38]:
tot_df = tot_df.groupby(['id', 'answer']).count().reset_index().rename(columns = {'annotator' : 'count'})

In [42]:
golden_df = tot_df[tot_df['count'] ==4]

In [47]:
addon_df = tot_df[(tot_df['answer'] == '<Difficulty Level>2</Difficulty Level>') & (tot_df['count'] == 3)]

In [48]:
golden_df = pd.concat([golden_df, addon_df], axis = 0)

In [49]:
golden_df

,id,answer,count
0,70180459,<Difficulty Level>0</Difficulty Level>,4
3,70243353,<Difficulty Level>1</Difficulty Level>,4
6,70266683,<Difficulty Level>1</Difficulty Level>,4
7,70267489,<Difficulty Level>1</Difficulty Level>,4
12,70373538,<Difficulty Level>1</Difficulty Level>,4
...,...,...,...
442,78157548,<Difficulty Level>2</Difficulty Level>,3
447,78269706,<Difficulty Level>2</Difficulty Level>,3
457,78367878,<Difficulty Level>2</Difficulty Level>,3
493,78812288,<Difficulty Level>2</Difficulty Level>,3


In [50]:
df_golden = pd.merge(df_q[['id', 'question']], golden_df[['id', 'answer']], on = 'id')

In [51]:
df_golden

,id,question,answer
0,71389500,<Title>Kubernetes: Error loading ASGI app. Att...,<Difficulty Level>1</Difficulty Level>
1,72118859,<Title>opencv read transperant logo</Title>. <...,<Difficulty Level>1</Difficulty Level>
2,72422859,<Title>Remove lines containing numbers attache...,<Difficulty Level>0</Difficulty Level>
3,70266683,<Title>Replace old records while inserting new...,<Difficulty Level>1</Difficulty Level>
4,72329302,<Title>How to write a FAST API function taking...,<Difficulty Level>1</Difficulty Level>
...,...,...,...
118,78310990,<Title>LightGBM with Multi-Output Regression a...,<Difficulty Level>1</Difficulty Level>
119,71698879,<Title>Why does a list appear as a comment wit...,<Difficulty Level>1</Difficulty Level>
120,70731352,<Title>Python str not list</Title>. <Question>...,<Difficulty Level>0</Difficulty Level>
121,74047007,<Title>How to detect black contour in image us...,<Difficulty Level>1</Difficulty Level>


In [ ]:
file_path = f"{path_list['data_root_dir']}/result/annotate_difficulty"  

df_golden.to_csv(f'{file_path}/q_output_code_y_snapshot2_md.csv', index=False)
# /mnt/hdd/mghan/so_difficulty_measure/result/annotate_difficulty/q_output_code_y_snapshot2_md.csv

